[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/analysis_multi-animal.ipynb)

# 📓 Notebook 3 – Analysis of multi animal (top-view mouse)
## 1. Introduction & objectives

In this notebook, you will analyze pose estimation outputs generated with the SuperAnimal ModelZoo on a top view multi animal video containing several mice.

**Learning goals:**

After this notebook, you should be able to:
- Load and preprocess multi animal pose data from SuperAnimal DLC
- Implement your own filtering and interpolation choices
- Compute activity and social metrics per mouse
- Integrate the results into a summary table

--- 

**About this notebook**

In this notebook, you will analyze pose-estimation data from freely-moving mice. 

# 🐭🐭🏠🎥 The Mouse House: multi animal pose challenge

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/illustrations/cover-mice.png" width="50%">

Five mice live together in the "Mouse House" 🐭🤍🏠, a fully monitored arena.
Every movement is tracked with SuperAnimal DeepLabCut.

Your task is to use pose data to build a behavioral profile for each mouse:
- Who is the Hyperactive One?
- Who is the Social Butterfly?
- Who is the Lone Wolf?
- And who wins each "medal" category?

🥇🥈🥉 At the end, you will assign gold, silver, and bronze medals in:
- Activity
- Sociability

For this exercise, you will work mostly independently, but everyone must create the same output variable names and structure so we can compare results.

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/single-frame-multi.png" width="50%">

--- 
**Instructions**

This notebook mixes pre-filled code cells (ready to run) and coding exercises that you will complete.

- Some cells are already complete (just run them).
- When you see a cell with a TODO, you must write code.
- You are free to choose methods, but you must respect:
  - Input: the provided pose file
  - Output variable names and column names as indicated. 

👉 Here’s how to work through it:
1. Read carefully each section before running the cells.
2. When a cell requires you to code, you’ll see a TODO comment.
3. The TODO will tell you how many lines of code you are expected to write.
4. Write your code only between the markers:
    
```python
# >>>>>>>>>>>>>>>>>>>
# your code goes here
# <<<<<<<<<<<<<<<<<<<
```

✋ Do not edit anything outside these markers.

⚡ After finishing the course, feel free to experiment and modify the notebook as you like!

---




## 2. Data Loading & Format Inspection

### 2.1 Download data (prefilled)

**📋 Instructions:**
- Run the code cell below to download the dataset file.

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# mice-5_5min
FILE_ID = "15Eiib-vpdunmxzYiP3RfKL0_iur-dCSJ"

URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# --- Load the cleaned H5 file into a pandas DataFrame ---
df = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:8], "...")

In [ ]:
# Load the HDF5 pose output into a pandas DataFrame.

def read_pose_h5(path: Path) -> pd.DataFrame:
    """
    Reads a DeepLabCut output file (.h5) and returns a pandas DataFrame.
    DLC can save results under different internal keys (e.g. 'df', 'tracks'),
    so this function tries several known keys until one works.
    """
    for key in ("df_with_missing", "df", "tracks", "pose"):
        try:
            return pd.read_hdf(path, key=key)
        except Exception:
            pass # Try next possible key if this one doesn't exist
        
    # If no specific key worked, try default    
    return pd.read_hdf(path)

# ---- Load the HDF5 pose output into a pandas DataFrame ----
df = read_pose_h5(DEST)

# ---- Basic sanity check ----

# ---- STUDENT TASK ----
# >>>>>>>>>>>>>>>>>>>
# 🧩 TODO: Verify that the DataFrame "df" has at least one row.
# Hint: use an assert statement to ensure the condition is True.
# Pseudo-example: ( assert <logical statement>, “message to return if assertion fails” )
#   If the file was loaded but empty, raise an error.
#   If not empty, the notebook will continue silently..
# YOUR CODE HERE ↓↓↓ : assert ...
assert df.shape[0] > 0, "The DataFrame is empty. Check if the file loaded correctly."
# <<<<<<<<<<<<<<<<<<<


# ---- TEACHER TEST ----
def test_assertion():
    """Check that the student's assertion refers to the right variable and logic."""
    try:
        assert isinstance(df, pd.DataFrame), "df is not a DataFrame"
        assert df.shape[0] > 0, "DataFrame seems empty"
        print("✅ Test passed, correct assertion and valid DataFrame.")
    except AssertionError as e:
        print("⚠️ Test failed:", e)

test_assertion()


print("✅ H5 loaded successfully!.")

# Show dataframe info
print(" Data shape (rows, columns):", df.shape)
# Display the first 5 rows as a nice HTML table
display(df)  # Display first 5 rows as a formatted HTML table

In [ ]:
# 👉🏼 PREFILLED CELL — JUST RUN IT

# Extract names from the column multi-index
animals = df.columns.get_level_values("individuals").unique()
bodyparts = df.columns.get_level_values("bodyparts").unique()

print("Number of animals:", len(animals))
print("Animals:", animals.tolist())
print("Number of bodyparts:", len(bodyparts))
print("Bodyparts:", bodyparts.tolist())

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Helper to summarize per-animal signal quality & motion

def animal_activity_summary(df: pd.DataFrame, conf_thresh: float = 0.5) -> pd.DataFrame:
    """
    Returns a small per-animal table with:
      - mean_likelihood : mean over all bodyparts/frames (often -1 when unused)
      - frac_conf       : fraction of points with likelihood >= conf_thresh (ignores <0)
      - mean_xy_var     : average variance of x/y where detections exist
    Sorted so the most likely real animal is on top.
    """
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns (scorer/individuals/bodyparts/coords).")
    expected = ['scorer', 'individuals', 'bodyparts', 'coords']
    if list(df.columns.names) != expected:
        raise ValueError(f"Unexpected column levels: {df.columns.names} (expected {expected})")

    idx = pd.IndexSlice
    animals = df.columns.get_level_values("individuals").unique()

    rows = []
    for a in animals:
        A = df.xs(a, axis=1, level="individuals")

        # Likelihoods table: (frames, bodyparts)
        L = A.xs("likelihood", axis=1, level="coords")
        mean_L = float(L.where(L >= 0).mean().mean())

        # Valid (>=0) then fraction above threshold
        L_valid = L.where(L >= 0)
        frac_conf = float((L_valid >= conf_thresh).mean().mean())

        # Build masked XY (only where L is valid) to get motion variance
        XY = A.loc[:, idx[:, :, ["x", "y"]]]  # (frames, bodyparts, coords[x,y])

        det_mask = L_valid.notna()  # (frames, bodyparts)
        # duplicate mask for x and y, then reorder levels to match XY
        mask_xy = pd.concat([det_mask, det_mask], axis=1, keys=["x", "y"])
        mask_xy = mask_xy.swaplevel(0, 2, axis=1).swaplevel(0, 1, axis=1).sort_index(axis=1)
        mask_xy = mask_xy.reindex(columns=XY.columns)

        mov_var = float(XY.where(mask_xy).var(ddof=0).mean())

        rows.append((a, mean_L, frac_conf, mov_var))

    out = (pd.DataFrame(rows, columns=["animal", "mean_likelihood", "frac_conf", "mean_xy_var"])
             .set_index("animal")
             .sort_values(["frac_conf", "mean_xy_var", "mean_likelihood"], ascending=False))
    return out

In [ ]:
# PREFILLED CELL - COMPLETE THE TODO THEN RUN

print("\n=== Detecting the most likely real animal... ===")

# >>>>>>>>>>>>>>>>>>>
# 🧩 TODO: compute the per-animal summary using the helper (1 line). Try conf_thresh=0.5 first.
# YOUR CODE (1 line) HERE ↓↓↓ : 
# summary = ...
summary = animal_activity_summary(df, conf_thresh=0.5)
# <<<<<<<<<<<<<<<<<<


# --- Display result ---
print("\n--- Active-animal summary (sorted) ---")
display(summary)